In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="pnnbao-ump/VieNeu-TTS-140h", 
    repo_type="dataset", local_dir="./VieNeu-TTS-140h", allow_patterns="*.arrow")

Fetching 49 files: 100%|██████████| 49/49 [01:22<00:00,  1.68s/it]


'/home/ubuntu/VieNeu-TTS-140h'

In [11]:
files = glob('VieNeu-TTS-140h/*.arrow')
len(files)

49

In [15]:
import pyarrow as pa

def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        with open(f, "rb") as f:
            reader = pa.ipc.open_stream(f)
            table = reader.read_all()
        df = table.to_pandas()
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker'].iloc[i]}"
            })
        
    return data

In [19]:
# data = multiprocessing(files, loop, cores = 20)

In [18]:
len(data)

73882

In [20]:
with open('VieNeu-TTS-140h.json', 'w') as fopen:
    json.dump(data, fopen)

In [21]:
audio_files = [d['audio_filename'] for d in data]

with open('VieNeu-TTS-140h-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [24]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'VieNeu-TTS-140h_audio/VieNeu-TTS-140h-data-00048-of-00049.arrow_0.mp3',
 'text': 'Đáp trả yêu cầu soi vào chiếc gương của trẻ chính là cột mốc của việc trở thành cha mẹ tỉnh thức.',
 'speaker': 'VieNeu-TTS-140h_audio_jellyfish1010_1080'}

In [28]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'VieNeu-TTS-140h')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 25.27ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  92%|█████████▏| 6.25MB / 6.77MB, 31.3MB/s  
Processing Files (1 / 1): 100%|██████████| 6.77MB / 6.77MB, 19.7MB/s  
Processing Files (1 / 1): 100%|██████████| 6.77MB / 6.77MB, 16.9MB/s  
New Data Upload: 100%|██████████| 6.77MB / 6.77MB, 16.9MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.25 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/94a2354b562529a1ee59fd9aea0ba5cc12db70bb', commit_message='Upload dataset', commit_description='', oid='94a2354b562529a1ee59fd9aea0ba5cc12db70bb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [26]:
# !zip -rq VieNeu-TTS-140h_audio.zip VieNeu-TTS-140h_audio

In [27]:
# !hf upload malaysia-ai/Multilingual-TTS VieNeu-TTS-140h_audio.zip --repo-type=dataset

In [ ]:
# !zip -rq VieNeu-TTS-140h_audio_neucodec.zip VieNeu-TTS-140h_audio_neucodec

In [ ]:
# !hf upload malaysia-ai/Multilingual-TTS VieNeu-TTS-140h_audio_neucodec.zip --repo-type=dataset